### 1. imports

**Nesta etapa, importamos as bibliotecas fundamentais como `pandas` para manipulação dos dados tabulares e arquivos no processo de ETL.**

In [ ]:
import pandas as pd
import glob
import os
from functools import reduce
import numpy as np

## 2. etl

### 2.1 estatística educação básica

**Definimos a variável contendo o caminho relativo/absoluto do arquivo alvo estático, visando organizar melhor o fluxo de dados brutos.**

In [ ]:
caminho_arquivo_entrada = "/home/raimundoivy/Documents/pad_avaliação_02/dados/Sinopse_Estatistica_da_Educação_Basica_2024.xlsx"

**Criamos nossa função isolada de ETL (Extração, Transformação e Carga).**
Esta função foca na extração e limpeza da fonte de **educação rural**, derretendo colunas em nível de dependência.

In [5]:
def extrair_transformar_carregar_planilha_educacao(caminho_arquivo_entrada):

    dataframe_educacao_bruto = pd.read_excel(
        caminho_arquivo_entrada, sheet_name="1.26", skiprows=9, header=None
    )

    dataframe_limpo = dataframe_educacao_bruto[[3, 10, 11, 12, 13, 14]].copy()
    dataframe_limpo.columns = [
        "codigo_municipio",
        "rural_total",
        "rural_federal",
        "rural_estadual",
        "rural_municipal",
        "rural_privado",
    ]

    dataframe_limpo = dataframe_limpo[
        pd.to_numeric(dataframe_limpo["codigo_municipio"], errors="coerce").notnull()
    ]
    dataframe_limpo["codigo_municipio"] = dataframe_limpo["codigo_municipio"].astype(
        int
    )

    dataframe_derretido = dataframe_limpo.melt(
        id_vars=["codigo_municipio"],
        value_vars=[
            "rural_federal",
            "rural_estadual",
            "rural_municipal",
            "rural_privado",
        ],
        var_name="dependencia_administrativa",
        value_name="matriculas_ensino_medio_rural",
    )

    dataframe_derretido["dependencia_administrativa"] = dataframe_derretido[
        "dependencia_administrativa"
    ].str.replace("rural_", "", regex=True)
    dataframe_derretido["matriculas_ensino_medio_rural"] = pd.to_numeric(
        dataframe_derretido["matriculas_ensino_medio_rural"], errors="coerce"
    )

    dataframe_derretido = dataframe_derretido.dropna(
        subset=["matriculas_ensino_medio_rural"]
    )
    dataframe_derretido["matriculas_ensino_medio_rural"] = dataframe_derretido[
        "matriculas_ensino_medio_rural"
    ].astype(int)

    return dataframe_derretido

**Em seguida, rodamos a função de ETL previamente configurada sobre o dataset, já visualizando os primeiros 10 registros extraídos do formato limpo.**

In [ ]:
dataframe_educacao_final = extrair_transformar_carregar_planilha_educacao(
    caminho_arquivo_entrada
)
display(dataframe_educacao_final.head(10))

,codigo_municipio,dependencia_administrativa,matriculas_ensino_medio_rural
0,1100015,federal,0
1,1100379,federal,0
2,1100403,federal,0
3,1100346,federal,0
4,1100023,federal,567
5,1100452,federal,0
6,1100031,federal,0
7,1100601,federal,0
8,1100049,federal,0
9,1100700,federal,0


**No encerramento dessa seção de dados, nós exportamos o dataframe tratado para um sub-arquivo `.csv` físico que alimenta nossa futura engrenagem mestre (Master).**

In [ ]:
caminho_arquivo_csv_saida = "/home/raimundoivy/Documents/pad_avaliação_02/dados/educacao_basica_2024_1_26_cleaned.csv"
print(f"\nsalvando em {caminho_arquivo_csv_saida }...")
dataframe_educacao_final.to_csv(
    caminho_arquivo_csv_saida, index=False, encoding="utf-8"
)
print("concluído!")


salvando em /home/raimundoivy/Documents/pad_avaliação_02/dados/educacao_basica_2024_1_26_cleaned.csv...
concluído!


### 2.2 tabela6873 -- tratores

**Definimos a variável contendo o caminho relativo/absoluto do arquivo alvo estático, visando organizar melhor o fluxo de dados brutos.**

In [ ]:
caminho_arquivo_entrada = (
    "/home/raimundoivy/Documents/pad_avaliação_02/dados/tabela6873.csv"
)

**Criamos nossa função isolada de ETL (Extração, Transformação e Carga).**
Esta função cuida da filtragem do dado mecânico agrícola, desempilhando os contadores de propriedades x veículos.

In [ ]:
def extrair_transformar_carregar_tabela_tratores(caminho_arquivo_entrada):
    print("carregando csv bruto...")
    dataframe_bruto = pd.read_csv(
        caminho_arquivo_entrada,
        sep=";",
        skiprows=4,
        engine="python",
        na_values=["-", "X", "..", "...", " "],
        on_bad_lines="skip",
        decimal=",",
    )
    dataframe_bruto.columns = [
        "codigo_municipio",
        "nome_localidade",
        "condicao_produtor",
        "codigo_filtro",
        "grupos_atividade",
        "valor_registrado",
    ]
    dataframe_bruto = dataframe_bruto.dropna(subset=["codigo_municipio"])

    dataframe_bruto["is_ibge_numerico"] = (
        dataframe_bruto["codigo_municipio"].astype(str).str.isnumeric()
    )
    linhas_validas = dataframe_bruto[dataframe_bruto["is_ibge_numerico"]].copy()

    numero_de_registros = len(linhas_validas) // 2

    dataframe_estabelecimentos_dividido = linhas_validas.iloc[
        :numero_de_registros
    ].copy()
    dataframe_maquinas_dividido = linhas_validas.iloc[numero_de_registros:].copy()

    print("processando valores numéricos...")
    dataframe_estabelecimentos_dividido["numero_de_estabelecimentos_agricolas"] = (
        pd.to_numeric(
            dataframe_estabelecimentos_dividido["valor_registrado"], errors="coerce"
        )
    )
    dataframe_maquinas_dividido["numero_de_tratores_em_uso"] = pd.to_numeric(
        dataframe_maquinas_dividido["valor_registrado"], errors="coerce"
    )

    chaves_mesclagem = [
        "codigo_municipio",
        "nome_localidade",
        "condicao_produtor",
        "grupos_atividade",
        "codigo_filtro",
    ]
    dataframe_maquinas_enxuto = dataframe_maquinas_dividido[
        chaves_mesclagem + ["numero_de_tratores_em_uso"]
    ]

    print("mesclando blocos de variáveis...")
    dataframe_final = pd.merge(
        dataframe_estabelecimentos_dividido,
        dataframe_maquinas_enxuto,
        on=chaves_mesclagem,
        how="left",
    )

    dataframe_final["sigla_estado"] = dataframe_final["nome_localidade"].str.extract(
        r"\((.*?)\)$"
    )[0]
    dataframe_final["nome_municipio"] = dataframe_final["nome_localidade"].str.replace(
        r"\s*\(.*?\)$", "", regex=True
    )

    dataframe_final["nivel_geografico"] = (
        dataframe_final["codigo_municipio"].astype(str).str.len()
    )

    colunas_para_remover = [
        "condicao_produtor",
        "grupos_atividade",
        "nivel_geografico",
        "codigo_filtro",
        "valor_registrado",
        "is_ibge_numerico",
        "nome_localidade",
    ]
    dataframe_final = dataframe_final.drop(
        columns=[c for c in colunas_para_remover if c in dataframe_final.columns],
        errors="ignore",
    )

    colunas_esperadas = [
        "codigo_municipio",
        "sigla_estado",
        "nome_municipio",
        "numero_de_estabelecimentos_agricolas",
        "numero_de_tratores_em_uso",
    ]
    dataframe_final = dataframe_final[
        [c for c in colunas_esperadas if c in dataframe_final.columns]
    ]

    dataframe_final["codigo_municipio"] = dataframe_final["codigo_municipio"].astype(
        int
    )

    return dataframe_final

**Em seguida, rodamos a função de ETL previamente configurada sobre o dataset, já visualizando os primeiros 10 registros extraídos do formato limpo.**

In [ ]:
dataframe_tratores_final = extrair_transformar_carregar_tabela_tratores(
    caminho_arquivo_entrada
)
print(f"\netl concluído. formato final: {dataframe_tratores_final .shape }")
display(dataframe_tratores_final.head())

carregando csv bruto...
processando valores numéricos...
mesclando blocos de variáveis...

etl concluído. formato final: (5569, 5)


,codigo_municipio,sigla_estado,nome_municipio,numero_de_estabelecimentos_agricolas,numero_de_tratores_em_uso
0,1,NaN,Brasil,734280.0,1229907.0
1,1,NaN,Norte,35092.0,58436.0
2,2,NaN,Nordeste,53284.0,83866.0
3,3,NaN,Sudeste,208791.0,373952.0
4,4,NaN,Sul,347476.0,517042.0


**No encerramento dessa seção de dados, nós exportamos o dataframe tratado para um sub-arquivo `.csv` físico que alimenta nossa futura engrenagem mestre (Master).**

In [ ]:
caminho_arquivo_csv_saida = (
    "/home/raimundoivy/Documents/pad_avaliação_02/dados/tabela6873_cleaned.csv"
)
print(f"\nsalvando em {caminho_arquivo_csv_saida }...")
dataframe_tratores_final.to_csv(
    caminho_arquivo_csv_saida, index=False, encoding="utf-8"
)
print("concluído!")


salvando em /home/raimundoivy/Documents/pad_avaliação_02/dados/tabela6873_cleaned.csv...
concluído!


### 2.3 tabela6873 -- área em hectares

**Definimos a variável contendo o caminho relativo/absoluto do arquivo alvo estático, visando organizar melhor o fluxo de dados brutos.**

In [ ]:
caminho_arquivo_entrada = (
    "/home/raimundoivy/Documents/pad_avaliação_02/dados/tabela6773.csv"
)

**Criamos nossa função isolada de ETL (Extração, Transformação e Carga).**
O script aborda a limpeza da contabilidade da área física (em hectares) atrelada às entidades válidas do IBGE.

In [ ]:
def extrair_transformar_carregar_tabela_estabelecimentos(caminho_arquivo_entrada):
    print("carregando csv bruto...")

    dataframe_bruto = pd.read_csv(
        caminho_arquivo_entrada,
        sep=";",
        skiprows=4,
        engine="python",
        na_values=["-", "X", "..", "...", " "],
        on_bad_lines="skip",
        decimal=",",
    )

    dataframe_bruto.columns = [
        "codigo_municipio",
        "nome_localidade",
        "finalidade_uso_terra",
        "faixa_renda",
        "codigo_filtro",
        "status_associacao",
        "valor_registrado",
    ]

    dataframe_bruto = dataframe_bruto.dropna(subset=["codigo_municipio"])

    dataframe_bruto["is_ibge_numerico"] = (
        dataframe_bruto["codigo_municipio"].astype(str).str.isnumeric()
    )
    linhas_validas = dataframe_bruto[dataframe_bruto["is_ibge_numerico"]].copy()

    numero_de_registros = len(linhas_validas) // 2

    dataframe_estabelecimentos_dividido = linhas_validas.iloc[
        :numero_de_registros
    ].copy()
    dataframe_area_dividido = linhas_validas.iloc[numero_de_registros:].copy()

    print("processando colunas...")
    dataframe_estabelecimentos_dividido["numero_de_estabelecimentos_agricolas"] = (
        pd.to_numeric(
            dataframe_estabelecimentos_dividido["valor_registrado"], errors="coerce"
        )
    )
    dataframe_area_dividido["area_total_em_hectares"] = pd.to_numeric(
        dataframe_area_dividido["valor_registrado"], errors="coerce"
    )

    dataframe_estabelecimentos_dividido["sigla_estado"] = (
        dataframe_estabelecimentos_dividido["nome_localidade"].str.extract(
            r"\((.*?)\)$"
        )[0]
    )
    dataframe_estabelecimentos_dividido["nome_municipio"] = (
        dataframe_estabelecimentos_dividido["nome_localidade"].str.replace(
            r"\s*\(.*?\)$", "", regex=True
        )
    )

    dataframe_estabelecimentos_dividido["nivel_geografico"] = (
        dataframe_estabelecimentos_dividido["codigo_municipio"].astype(str).str.len()
    )

    dataframe_estabelecimentos_dividido = dataframe_estabelecimentos_dividido.drop(
        columns=["nome_localidade", "valor_registrado", "is_ibge_numerico"]
    )
    dataframe_area_dividido = dataframe_area_dividido[
        ["codigo_municipio", "area_total_em_hectares"]
    ]

    print("mesclando blocos de variáveis...")

    dataframe_final = pd.merge(
        dataframe_estabelecimentos_dividido,
        dataframe_area_dividido,
        on="codigo_municipio",
        how="left",
    )

    dataframe_final = dataframe_final[
        [
            "codigo_municipio",
            "nivel_geografico",
            "sigla_estado",
            "nome_municipio",
            "finalidade_uso_terra",
            "faixa_renda",
            "status_associacao",
            "numero_de_estabelecimentos_agricolas",
            "area_total_em_hectares",
        ]
    ]

    dataframe_final["codigo_municipio"] = dataframe_final["codigo_municipio"].astype(
        int
    )

    return dataframe_final

**Em seguida, rodamos a função de ETL previamente configurada sobre o dataset, já visualizando os primeiros 10 registros extraídos do formato limpo.**

In [ ]:
dataframe_estabelecimentos_final = extrair_transformar_carregar_tabela_estabelecimentos(
    caminho_arquivo_entrada
)
print(f"etl concluído. formato final: {dataframe_estabelecimentos_final .shape }")
display(dataframe_estabelecimentos_final.head())

carregando csv bruto...
processando colunas...
mesclando blocos de variáveis...
etl concluído. formato final: (5564, 9)


,codigo_municipio,nivel_geografico,sigla_estado,nome_municipio,finalidade_uso_terra,faixa_renda,status_associacao,numero_de_estabelecimentos_agricolas,area_total_em_hectares
0,1,1,NaN,Brasil,Total,Total,Total,5073324,351289816.0
1,1100015,7,RO,Alta Floresta D'Oeste,Total,Total,Total,2886,372746.0
2,1100023,7,RO,Ariquemes,Total,Total,Total,2928,334256.0
3,1100031,7,RO,Cabixi,Total,Total,Total,1075,113085.0
4,1100049,7,RO,Cacoal,Total,Total,Total,3814,221390.0


**No encerramento dessa seção de dados, nós exportamos o dataframe tratado para um sub-arquivo `.csv` físico que alimenta nossa futura engrenagem mestre (Master).**

In [ ]:
caminho_arquivo_csv_saida = (
    "/home/raimundoivy/Documents/pad_avaliação_02/dados/tabela6773_cleaned.csv"
)
print(f"\nsalvando em {caminho_arquivo_csv_saida }...")
dataframe_estabelecimentos_final.to_csv(
    caminho_arquivo_csv_saida, index=False, encoding="utf-8"
)
print("concluído!")


salvando em /home/raimundoivy/Documents/pad_avaliação_02/dados/tabela6773_cleaned.csv...
concluído!


### 2.4 tabela6873 -- produtos / área colhida

**Definimos a variável contendo o caminho relativo/absoluto do arquivo alvo estático, visando organizar melhor o fluxo de dados brutos.**

In [ ]:
caminho_arquivo_entrada = (
    "/home/raimundoivy/Documents/pad_avaliação_02/dados/tabela1612.csv"
)

**Para arquivos imensos em formato super largo, predefinimos as regras colunares contendo anos avaliados ao longo do tempo (ex: `1974` a `2024`) cruzados com lavouras principais.**

In [ ]:
anos = range(1974, 2025)
produtos = [
    "Algodão herbáceo (em caroço)",
    "Cana-de-açúcar",
    "Milho (em grão)",
    "Soja (em grão)",
]
columns = ["codigo_municipio", "nome_localidade"]
for indice_ano in anos:
    for indice_produto in produtos:
        columns.append(f"{indice_ano }_{indice_produto }")
print(f"esperado {len (columns )} columns.")

esperado 206 columns.


**Fazemos uso dinâmico de leitura nativa I/O na própria base para intuir com precisão quantas colunas a tabulação de fato obedece sob essa origem.**

In [ ]:
with open(caminho_arquivo_entrada, "r", encoding="utf-8") as f:
    for _ in range(5):
        next(f)
    primeira_linha_dados = f.readline().strip()
numero_de_colunas_dados = len(primeira_linha_dados.split(";"))
print(f"linhas de dados parecem ter {numero_de_colunas_dados } columns.")

linhas de dados parecem ter 206 columns.


**Descartando campos inexistentes oriundos de ponto-e-vírgula avulsos no final da formatação do `csv`.**

In [ ]:
colunas_para_usar = list(range(len(columns)))

**Agora rodamos o carregamento massivo através do `c-engine` com suporte de baixo uso de memória (`low_memory=False`), processando de forma performática.**

In [ ]:
dataframe = pd.read_csv(
    caminho_arquivo_entrada,
    sep=";",
    skiprows=5,
    names=columns,
    usecols=colunas_para_usar,
    na_values=["-", "X", "..", "...", " "],
    decimal=",",
    engine="c",
    low_memory=False,
)
print(f"formato do csv carregado: {dataframe .shape }")

formato do csv carregado: (11167, 206)


**Tratamos o código da região convertendo eventuais contaminações não numéricas (NaN, null, lixo) forçando a exclusão com o `errors='coerce'`.**

In [ ]:
dataframe = dataframe[
    pd.to_numeric(dataframe["codigo_municipio"], errors="coerce").notnull()
]
dataframe["codigo_municipio"] = dataframe["codigo_municipio"].astype(int)

**Construímos a demarcação quantitativa `nivel_geografico` calculando explicitamente quantos algarismos compõem o geocódigo.**

In [ ]:
dataframe["codigo_municipio_string"] = dataframe["codigo_municipio"].astype(str)
dataframe["nivel_geografico"] = dataframe["codigo_municipio_string"].str.len()

**Tratamos o código da região convertendo eventuais contaminações não numéricas (NaN, null, lixo) forçando a exclusão com o `errors='coerce'`.**

In [ ]:
variaveis_identificadoras = ["codigo_municipio", "nome_localidade", "nivel_geografico"]
variaveis_valor = [
    col for col in columns if col not in ["codigo_municipio", "nome_localidade"]
]
print("aplicando melt no dataframe...")
dataframe_derretido = dataframe.melt(
    id_vars=variaveis_identificadoras,
    value_vars=variaveis_valor,
    var_name="ano_e_produto",
    value_name="area_colhida_em_hectares",
)
print("extracting indice_ano and product...")
dataframe_derretido[["ano_referencia", "produto_agricola"]] = dataframe_derretido[
    "ano_e_produto"
].str.extract(r"^(\d{4})_(.*)$")
dataframe_derretido.drop(columns=["ano_e_produto"], inplace=True)
dataframe_derretido["ano_referencia"] = dataframe_derretido["ano_referencia"].astype(
    int
)
dataframe_derretido["area_colhida_em_hectares"] = pd.to_numeric(
    dataframe_derretido["area_colhida_em_hectares"], errors="coerce"
)
dataframe_derretido.dropna(subset=["area_colhida_em_hectares"], inplace=True)

aplicando melt no dataframe...
extracting indice_ano and product...


**Desejamos que o escopo de município tenha nome limpo e estrito; logo, as funções nativas de limpeza `Regex` (expressões regulares) fatiam a String extraindo sigla do Estado para a parte correspondente.**

In [ ]:
dataframe_derretido["sigla_estado"] = dataframe_derretido[
    "nome_localidade"
].str.extract(r"\((.*?)\)$")[0]
dataframe_derretido["nome_municipio"] = dataframe_derretido[
    "nome_localidade"
].str.replace(r"\s*\(.*?\)$", "", regex=True)

In [ ]:
dataframe_final = dataframe_derretido[
    [
        "codigo_municipio",
        "nivel_geografico",
        "sigla_estado",
        "nome_municipio",
        "ano_referencia",
        "produto_agricola",
        "area_colhida_em_hectares",
    ]
]
print(f"etl concluído. formato final: {dataframe_final .shape }")
display(dataframe_final.head())

etl concluído. formato final: (1065021, 7)


,codigo_municipio,nivel_geografico,sigla_estado,nome_municipio,ano_referencia,produto_agricola,area_colhida_em_hectares
0,1,1,NaN,Brasil,1974,Algodão herbáceo (em caroço),1726036.0
1,1,1,NaN,Norte,1974,Algodão herbáceo (em caroço),256.0
2,2,1,NaN,Nordeste,1974,Algodão herbáceo (em caroço),809101.0
3,3,1,NaN,Sudeste,1974,Algodão herbáceo (em caroço),496122.0
4,4,1,NaN,Sul,1974,Algodão herbáceo (em caroço),310000.0


**No encerramento dessa seção de dados, nós exportamos o dataframe tratado para um sub-arquivo `.csv` físico que alimenta nossa futura engrenagem mestre (Master).**

In [ ]:
caminho_arquivo_csv_saida = (
    "/home/raimundoivy/Documents/pad_avaliação_02/dados/tabela1612_cleaned.csv"
)
print(f"salvando em {caminho_arquivo_csv_saida }...")
dataframe_final.to_csv(caminho_arquivo_csv_saida, index=False, encoding="utf-8")
print("concluído!")

salvando em /home/raimundoivy/Documents/pad_avaliação_02/dados/tabela1612_cleaned.csv...
concluído!


### 2.4 ibge população

**Definimos a variável contendo o caminho relativo/absoluto do arquivo alvo estático, visando organizar melhor o fluxo de dados brutos.**

In [ ]:
caminho_arquivo_entrada = (
    "/home/raimundoivy/Documents/pad_avaliação_02/dados/br_ibge_populacao_municipio.csv"
)

**Criamos nossa função isolada de ETL (Extração, Transformação e Carga).**
A rotina garante a conversão e o expurgo das notas de rodapé dos dados de moradores/população das planilhas Censo.

In [28]:
def extrair_transformar_carregar_populacao(caminho_arquivo_entrada):
    print("carregando dados populacionais...")
    dataframe = pd.read_csv(caminho_arquivo_entrada, sep=",")

    rename_map = {
        "ano": "ano_referencia",
        "id_municipio": "codigo_municipio",
        "sigla_uf": "sigla_estado",
        "populacao": "populacao_total",
    }
    dataframe = dataframe.rename(columns=rename_map)

    print(f"formato original: {dataframe .shape }")
    print("anos disponíveis:", sorted(dataframe["ano_referencia"].unique()))

    dataframe = dataframe.dropna(subset=["codigo_municipio"])
    dataframe["codigo_municipio"] = dataframe["codigo_municipio"].astype(int)

    return dataframe

**Em seguida, rodamos a função de ETL previamente configurada sobre o dataset, já visualizando os primeiros 10 registros extraídos do formato limpo.**

In [ ]:
dataframe_populacao_final = extrair_transformar_carregar_populacao(
    caminho_arquivo_entrada
)
print(f"\netl concluído. formato final: {dataframe_populacao_final .shape }")
display(dataframe_populacao_final.head())

carregando dados populacionais...
formato original: (191099, 4)
anos disponíveis: [np.int64(1991), np.int64(1992), np.int64(1993), np.int64(1994), np.int64(1995), np.int64(1996), np.int64(1997), np.int64(1998), np.int64(1999), np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

etl concluído. formato final: (191098, 4)


,ano_referencia,sigla_estado,codigo_municipio,populacao_total
0,1991,RO,1100015,31981.0
1,1991,RO,1100023,83684.0
2,1991,RO,1100031,8173.0
3,1991,RO,1100049,79036.0
4,1991,RO,1100056,21607.0


**No encerramento dessa seção de dados, nós exportamos o dataframe tratado para um sub-arquivo `.csv` físico que alimenta nossa futura engrenagem mestre (Master).**

In [ ]:
caminho_arquivo_csv_saida = "/home/raimundoivy/Documents/pad_avaliação_02/dados/br_ibge_populacao_municipio_cleaned.csv"
print(f"\nsalvando em {caminho_arquivo_csv_saida }...")
dataframe_populacao_final.to_csv(
    caminho_arquivo_csv_saida, index=False, encoding="utf-8"
)
print("concluído!")


salvando em /home/raimundoivy/Documents/pad_avaliação_02/dados/br_ibge_populacao_municipio_cleaned.csv...
concluído!


## 3. cruzando os dados

In [ ]:
print("loading individual datasets...")
dataframe_populacao = pd.read_csv(
    "/home/raimundoivy/Documents/pad_avaliação_02/dados/br_ibge_populacao_municipio_cleaned.csv"
)
dataframe_educacao = pd.read_csv(
    "/home/raimundoivy/Documents/pad_avaliação_02/dados/educacao_basica_2024_1_26_cleaned.csv"
)
dataframe_agricultura = pd.read_csv(
    "/home/raimundoivy/Documents/pad_avaliação_02/dados/tabela1612_cleaned.csv"
)
dataframe_estabelecimentos = pd.read_csv(
    "/home/raimundoivy/Documents/pad_avaliação_02/dados/tabela6773_cleaned.csv"
)
dataframe_tratores = pd.read_csv(
    "/home/raimundoivy/Documents/pad_avaliação_02/dados/tabela6873_cleaned.csv"
)
print("applying cross-reference logic via 'codigo_municipio'...")

loading individual datasets...
applying cross-reference logic via 'codigo_municipio'...


In [ ]:
dataframe_base = dataframe_populacao[dataframe_populacao["ano_referencia"] == 2024][
    ["codigo_municipio", "sigla_estado", "populacao_total"]
].copy()
dataframe_base = dataframe_base.rename(columns={"populacao_total": "populacao_em_2024"})

**Procedemos com integrações laterais sequenciais (`Left Join`) utilizando apenas a chave primária Municipal em comum, a fim de unificar todos os domínios independentes de dados numa tabela primária.**

In [ ]:
aggregated_education_data = (
    dataframe_educacao.groupby("codigo_municipio")["matriculas_ensino_medio_rural"]
    .sum()
    .reset_index()
)
dataframe_base = pd.merge(
    dataframe_base, aggregated_education_data, on="codigo_municipio", how="left"
)

In [ ]:
ano_maximo = dataframe_agricultura["ano_referencia"].max()
dataframe_agricultura_recente = dataframe_agricultura[
    dataframe_agricultura["ano_referencia"] == ano_maximo
]

In [35]:
dataframe_agricultura_pivo = dataframe_agricultura_recente.pivot_table(
    index="codigo_municipio",
    columns="produto_agricola",
    values="area_colhida_em_hectares",
    aggfunc="sum",
).reset_index()

**Procedemos com integrações laterais sequenciais (`Left Join`) utilizando apenas a chave primária Municipal em comum, a fim de unificar todos os domínios independentes de dados numa tabela primária.**

In [ ]:
dataframe_agricultura_pivo.columns = ["codigo_municipio"] + [
    f"area_{c .split (' (')[0 ].replace ('-','_').replace (' ','_').lower ()}_{ano_maximo }"
    for c in dataframe_agricultura_pivo.columns
    if c != "codigo_municipio"
]
dataframe_base = pd.merge(
    dataframe_base, dataframe_agricultura_pivo, on="codigo_municipio", how="left"
)

**Procedemos com integrações laterais sequenciais (`Left Join`) utilizando apenas a chave primária Municipal em comum, a fim de unificar todos os domínios independentes de dados numa tabela primária.**

In [ ]:
if "nivel_geografico" in dataframe_estabelecimentos.columns:
    dataframe_estabelecimentos_municipio = dataframe_estabelecimentos[
        dataframe_estabelecimentos["nivel_geografico"] == 7
    ].drop_duplicates(subset=["codigo_municipio"])
else:
    dataframe_estabelecimentos_municipio = dataframe_estabelecimentos.drop_duplicates(
        subset=["codigo_municipio"]
    )
dataframe_estabelecimentos_municipio = dataframe_estabelecimentos_municipio[
    [
        "codigo_municipio",
        "numero_de_estabelecimentos_agricolas",
        "area_total_em_hectares",
    ]
]
dataframe_estabelecimentos_municipio.columns = [
    "codigo_municipio",
    "total_estabelecimentos_agricolas",
    "area_total_agricola_em_hectares",
]
dataframe_base = pd.merge(
    dataframe_base,
    dataframe_estabelecimentos_municipio,
    on="codigo_municipio",
    how="left",
)

**Procedemos com integrações laterais sequenciais (`Left Join`) utilizando apenas a chave primária Municipal em comum, a fim de unificar todos os domínios independentes de dados numa tabela primária.**

In [ ]:
dataframe_tratores_municipio = dataframe_tratores.drop_duplicates(
    subset=["codigo_municipio"]
)
dataframe_tratores_municipio = dataframe_tratores_municipio[
    ["codigo_municipio", "numero_de_tratores_em_uso"]
]
dataframe_base = pd.merge(
    dataframe_base, dataframe_tratores_municipio, on="codigo_municipio", how="left"
)
print(
    f"\nformato unificado final de dados (1 linha por município): {dataframe_base .shape }"
)
display(dataframe_base.head())


formato unificado final de dados (1 linha por município): (5570, 11)


,codigo_municipio,sigla_estado,populacao_em_2024,matriculas_ensino_medio_rural,area_algodão_herbáceo_2024,area_cana_de_açúcar_2024,area_milho_2024,area_soja_2024,total_estabelecimentos_agricolas,area_total_agricola_em_hectares,numero_de_tratores_em_uso
0,1100015,RO,22853.0,72,NaN,84.0,9664.0,11758.0,2886.0,372746.0,517.0
1,1100023,RO,108573.0,703,NaN,730.0,37593.0,109520.0,2928.0,334256.0,501.0
2,1100031,RO,5690.0,33,NaN,92.0,104069.0,224895.0,1075.0,113085.0,247.0
3,1100049,RO,97637.0,189,NaN,673.0,16302.0,43797.0,3814.0,221390.0,465.0
4,1100056,RO,16975.0,38,NaN,616.0,129356.0,265911.0,719.0,126686.0,303.0


**Gravamos o arquivo de intersecção geográfica final unificado de Municípios agregados em grão sumarizado.**

In [39]:
caminho_arquivo_csv_master_saida = (
    "/home/raimundoivy/Documents/pad_avaliação_02/dados/master_municipios_agrupado.csv"
)
print(f"\nsalvando em {caminho_arquivo_csv_master_saida } ...")
dataframe_base.to_csv(caminho_arquivo_csv_master_saida, index=False, encoding="utf-8")
print("concluído!")


salvando em /home/raimundoivy/Documents/pad_avaliação_02/dados/master_municipios_agrupado.csv ...
concluído!


## 4. master granular (dataframe_base expandida)
ao invés de agrupar tudo em apenas 5.570 linhas, criamos uma visão detalhada para atingir +1.5m de registros.

**Entrando na fase master granular (`+ 1.5M records`), pegamos a densa fatia de safras lavouristas e alocamos cópia.**

In [40]:
print("iniciando mescla granular com matriculas rurais...")
dataframe_agricultura_expandido = dataframe_agricultura.copy()

iniciando mescla granular com matriculas rurais...


In [41]:
dados_dependencias_educacao = dataframe_educacao[
    ["codigo_municipio", "dependencia_administrativa", "matriculas_ensino_medio_rural"]
].copy()

**Através de um `Outer Join`, alinhamos verticalmente a variável de matrículas, abrindo alas do cubo hiperdimensional de estatísticas sobre a localidade.**

In [ ]:
dataframe_grade_final = pd.merge(
    dataframe_agricultura_expandido,
    dados_dependencias_educacao,
    on="codigo_municipio",
    how="outer",
)

**Executamos a consolidação das informações agrupadas em suas devidas entidades originais.**

In [ ]:
dataframe_grade_final = pd.merge(
    dataframe_grade_final,
    dataframe_populacao[["codigo_municipio", "ano_referencia", "populacao_total"]],
    on=["codigo_municipio", "ano_referencia"],
    how="left",
)

**Executamos a consolidação das informações agrupadas em suas devidas entidades originais.**

In [ ]:
if "nivel_geografico" in dataframe_estabelecimentos.columns:
    dataframe_estabelecimentos_municipio = dataframe_estabelecimentos[
        dataframe_estabelecimentos["nivel_geografico"] == 7
    ].drop_duplicates(subset=["codigo_municipio"])
else:
    dataframe_estabelecimentos_municipio = dataframe_estabelecimentos.drop_duplicates(
        subset=["codigo_municipio"]
    )
dataframe_estabelecimentos_municipio = dataframe_estabelecimentos_municipio[
    [
        "codigo_municipio",
        "numero_de_estabelecimentos_agricolas",
        "area_total_em_hectares",
    ]
]
dataframe_estabelecimentos_municipio.columns = [
    "codigo_municipio",
    "total_estabelecimentos_agricolas",
    "area_total_agricola_em_hectares",
]

dataframe_grade_final = pd.merge(
    dataframe_grade_final,
    dataframe_estabelecimentos_municipio,
    on="codigo_municipio",
    how="left",
)

dataframe_tratores_municipio = dataframe_tratores.drop_duplicates(
    subset=["codigo_municipio"]
)[["codigo_municipio", "numero_de_tratores_em_uso"]]
dataframe_grade_final = pd.merge(
    dataframe_grade_final,
    dataframe_tratores_municipio,
    on="codigo_municipio",
    how="left",
)

**No encerramento dessa seção de dados, nós exportamos o dataframe tratado para um sub-arquivo `.csv` físico que alimenta nossa futura engrenagem mestre (Master).**

In [45]:
dataframe_grade_final["produto_agricola"] = dataframe_grade_final[
    "produto_agricola"
].fillna("n/a (apenas dados escolares/estáticos)")

print(f"granular shape: {dataframe_grade_final .shape }")

caminho_arquivo_csv_saida = (
    "/home/raimundoivy/Documents/pad_avaliação_02/dados/master_municipios.csv"
)
print(f"\nsalvando em {caminho_arquivo_csv_saida }...")
dataframe_grade_final.to_csv(caminho_arquivo_csv_saida, index=False, encoding="utf-8")
print("finalizado!")

granular shape: (4252902, 13)

salvando em /home/raimundoivy/Documents/pad_avaliação_02/dados/master_municipios.csv...
finalizado!
